# Moni Pipeline Demo

DB 테이블 기반 AI 엔진 데모 (v0.2 — 다건 챌린지 + streak).

**v0.2 변경점**:
- 단건 → **최대 4건** 챌린지 반환 (압박도 상위 3개 + streak 보너스 1개)
- streak형 챌린지 추가 (무지출 연속 시 보너스)
- BE 직장인 더미데이터(`user-office-001`) 적용
- 카테고리: 카페 / 식비 / 의류 / 화장품 / 가전

In [ ]:
# Colab에서 실행 시
# !pip install prophet pandas numpy --quiet

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import json
from datetime import date
import pandas as pd

## 1. 시드 데이터 로드

In [ ]:
SEED_DIR = Path.cwd().parent / "seed_data"

transactions_df = pd.read_csv(SEED_DIR / "seed_transactions.csv")
users_df = pd.read_csv(SEED_DIR / "seed_users.csv")
category_settings_df = pd.read_csv(SEED_DIR / "seed_category_settings.csv")

print(f"transactions: {len(transactions_df)} rows")
print(transactions_df['final_category'].value_counts())

## 2. AI 엔진 호출 (다건 반환)

`get_today_challenges()`는 이제 **list**를 반환한다. 백엔드는 이 list를 row별로 `Daily_Challenges`에 INSERT.

In [ ]:
from moni_engine.engine import get_today_challenges

user_profile = users_df.iloc[0].to_dict()
target_date = date(2025, 11, 15)  # 데이터 범위(2025) 안의 날짜

results = get_today_challenges(
    transactions_df=transactions_df,
    user_profile=user_profile,
    category_settings_df=category_settings_df,
    target_date=target_date,
)

print(f"생성된 챌린지 수: {len(results)}\n")
for c in results:
    print(json.dumps(c, ensure_ascii=False, indent=2, default=str))
    print("-" * 50)

## 3. 챌린지 카드 미리보기

In [ ]:
from IPython.display import HTML, display

diff_color = {"Easy": "#2e7d32", "Medium": "#ef6c00", "Medium-Hard": "#e64a19",
              "Hard": "#d32f2f", "Special": "#6a1b9a"}

cards = ""
for c in results:
    md = c["ai_metadata"]
    color = diff_color.get(c["difficulty"], "#555")
    origin_badge = "⭐ BONUS" if md.get("challenge_origin") == "streak" else ""
    cards += f"""
    <div style='font-family:-apple-system,sans-serif; display:inline-block; vertical-align:top;
                width:300px; border:1px solid #e6e6e6; border-radius:16px; padding:16px;
                margin:6px; box-shadow:0 2px 8px rgba(0,0,0,0.04); background:white;'>
        <div style='display:flex; justify-content:space-between; align-items:center; margin-bottom:10px;'>
            <span style='background:{color}; color:white; padding:4px 10px; border-radius:999px;
                         font-size:11px; font-weight:700;'>{c['difficulty']} · {c['challenge_type']}</span>
            <span style='color:#6a1b9a; font-size:11px; font-weight:700;'>{origin_badge}</span>
        </div>
        <div style='font-size:15px; font-weight:700; margin-bottom:8px; line-height:1.4;'>{c['challenge_text']}</div>
        <div style='font-size:12px; color:#888;'>{c['category_name']} · +{c['xp_reward']} XP</div>
    </div>
    """
display(HTML(f"<div>{cards}</div>"))

## 4. 후보 카테고리 비교 (첫 챌린지 메타에 첨부됨)

In [ ]:
if results:
    eval_df = pd.DataFrame(results[0]["ai_metadata"]["evaluated_categories"])
    display(eval_df)

## 5. streak 발동 시나리오

직장인 데이터는 카페를 거의 매일 마셔서 streak이 잘 안 생긴다.
카페를 며칠 끊은 사용자를 구성하면 streak 보너스가 발동한다.

In [ ]:
rows = []
base = pd.Timestamp("2025-10-01")
for i in range(45):
    d = base + pd.Timedelta(days=i)
    ds = d.strftime("%Y-%m-%d")
    if d < pd.Timestamp("2025-11-11"):  # 마지막 4일 카페 끊음
        rows.append({"tx_id": f"c{i}", "user_id": "u1", "tx_date": ds, "tx_time": "08:30:00",
                     "amount": 4000, "merchant_name": "메가커피", "mydata_category": "카페",
                     "final_category": "카페", "is_user_corrected": False})
    rows.append({"tx_id": f"f{i}", "user_id": "u1", "tx_date": ds, "tx_time": "12:30:00",
                 "amount": 10000, "merchant_name": "구내식당", "mydata_category": "식비",
                 "final_category": "식비", "is_user_corrected": False})

demo_tx = pd.DataFrame(rows)
demo_cat = pd.DataFrame([
    {"id": 1, "user_id": "u1", "category_name": "카페", "budget_limit": 100000,
     "is_daily_challenge": True, "alert_threshold": 0.8},
    {"id": 2, "user_id": "u1", "category_name": "식비", "budget_limit": 400000,
     "is_daily_challenge": True, "alert_threshold": 0.9},
])
demo_user = {"user_id": "u1", "valid_data_start_date": None}

streak_results = get_today_challenges(demo_tx, demo_user, demo_cat, date(2025, 11, 15))
for c in streak_results:
    origin = c["ai_metadata"].get("challenge_origin")
    print(f"[{origin:8s}] {c['category_name']} | {c['challenge_type']} | XP{c['xp_reward']} | {c['challenge_text']}")